# Machine Translation using Transformers

In [20]:
import os.path
from typing import Any

import torch
import numpy as np
import spacy
import pandas as pd
import sklearn
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import os

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.mps.is_available() else "cpu"
)

## English to Hindi

In [2]:
dir_name = "indic_languages_corpus/bilingual/hi-en"

In [3]:
english = None
with open(os.path.join(os.path.abspath(dir_name), "train.en")) as f:
    english = f.readlines()

In [4]:
hindi = None
with open(os.path.join(os.path.abspath(dir_name), "train.hi")) as f:
    hindi = f.readlines()

In [5]:
X_train = english

In [6]:
y_train = hindi

##### Why not byte pair encoding or character by character or turn words into graph for autocomplete, or sentence translation or generation?

**We are designing infinte window transformers, handling millions or billions of sequences, we can easily do this**
###### Can we turn the data into graphs, since relationship between words is sparse same as wordnet

### Preprocessing

In [7]:
from collections import defaultdict, Counter

In [29]:
sub_corpus = X_train[:2]


def tokenize(text):
    vocabulary = Counter()
    corpus = [a.split() for a in text]

    for token in corpus:
        vocabulary.update([t for t in token])
    word_to_index = {word: i + 1 for i, (word, _) in enumerate(vocabulary.items())}
    word_to_index["pad"] = 0
    numerical_sequences = [
        [word_to_index[token] for token in tokens] for tokens in corpus
    ]
    max_length = max(len(seq) for seq in numerical_sequences)

    padded_sequences = [
        seq + [word_to_index["pad"]] * (max_length - len(seq))
        for seq in numerical_sequences
    ]
    return padded_sequences, vocabulary, word_to_index

In [30]:
padded_sequences, vocabulary_train, train_index = tokenize(X_train)

In [31]:
train_sequences, vocabulary, word_index_train = (
    padded_sequences,
    vocabulary_train,
    train_index,
)

In [32]:
target_sequences, vocabulary_target, target_index = tokenize(y_train)

In [33]:
test_sequences, vocabulary_test, word_index_test = (
    target_sequences,
    vocabulary_target,
    target_index,
)

In [34]:
len(train_sequences)

84557

In [35]:
len(vocabulary_train)

49762

In [36]:
len(vocabulary_test)

40396

In [37]:
vocabulary_train["pad"] = 0
vocabulary_target["pad"] = 0

In [38]:
vocabulary = vocabulary_train
vocabulary_test = vocabulary_target

In [39]:
len(word_index_test)

40397

In [40]:
# vocabulary_target.most_common(40)

In [41]:
vocabulary.most_common(10)

[('the', 14550),
 ('I', 12912),
 ('to', 11695),
 ('you', 11340),
 ('a', 9463),
 ('-', 8610),
 ('of', 5849),
 ('is', 4615),
 ('and', 4566),
 ('in', 4541)]

In [42]:
torch.mps.is_available()

True

In [43]:
X_train[:2]

['And what is their Sigil?\n', 'I do not want to die.\n']

In [44]:
y_train[:2]

['और उनके Sigil क्या है?\n', 'मैं मरना नहीं चाहता.\n']

In [45]:
train_sequences[:2]

[[1,
  2,
  3,
  4,
  5,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [6,
  7,
  8,
  9,
  10,
  11,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]]

In [46]:
len(vocabulary_train), len(vocabulary_test), len(word_index_train), len(word_index_test)

(49763, 40397, 49763, 40397)

In [48]:
train_sequences[:2]

[[1,
  2,
  3,
  4,
  5,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [6,
  7,
  8,
  9,
  10,
  11,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]]

In [24]:
VOCAB_SIZE = len(vocabulary)
BATCH_SIZE = 64

embedding_dim = 256
# no of GRUs
units = 1024
vocab_inp_size = len(train_index)
vocabulary_target_size = len(target_index)

In [25]:
VOCAB_SIZE

42897

In [49]:
X_train = torch.tensor(train_sequences, dtype=torch.long)
y_train = torch.tensor(target_sequences, dtype=torch.long)

In [29]:
# import pandas as pd
# df = pd.DataFrame.from_dict({'sentences':corpus,'numerical':numerical_sequences})

In [30]:
# df

### Encoder Decoder Architecture

In [27]:
class AttentionHead(nn.Module):
    """
    Multi Attention Head
    Splits Input in Q,K,V

    """

    def __init__(self, model_dim, H, dropout_rate=0.1):
        super().__init__()
        self.Wq = nn.Linear(model_dim, model_dim)
        self.Wk = nn.Linear(model_dim, model_dim)
        self.Wv = nn.Linear(model_dim, model_dim)
        self.H = H
        self.d_h = int(model_dim / H)
        self.dropout = nn.Dropout(p=dropout_rate)

        self.Wo = nn.Linear(model_dim, model_dim)

    def forward(self, sequences, attn_mask=False):
        """Input shape: [batch_size, seq_len, d_model=num_head * d_head]
        # if key_value_states are provided this layer is used as a cross-attention layer for text translation..

        # for the decoder

        """
        batch_size, seq_len, model_dim = sequences.size()
        Q = self.Wq(sequences)

        K, V = self.Wk(sequences), self.Wv(sequences)

        A = Q @ K.transpose(-2, -1)
        if attn_mask is not None and attn_mask:
            A = A.masked_fill(attn_mask == 0, -float("inf"))
        A = F.softmax(A / self.d_h**0.5, dim=-1)  # Applying softmax
        A = self.dropout(A)  # Final

        # Output Z

        Z = A @ V  # torch,tensor
        print(f"{Z.shape=}")
        ### Concatenating in parallel along heads and sequences
        ## Because continuous input
        # Z = (Z.contiguous().view(
        #     batch_size,seq_len,self.H*self.d_h)
        # )
        # 2. Transpose to move seq_len before H: shape (batch_size, seq_len, H, d_h)
        # NOTE: Z is now NON-CONTIGUOUS in memory!
        Z = Z.transpose(1, 2).reshape(batch_size, seq_len, model_dim)
        # 3. Concatenate all heads into d_model = H * d_h
        # Fails without .contiguous():
        # Z = Z.contiguous().view(batch_size,seq_len,self.H*self.d_h)
        # final linear projections
        Z = self.Wo(Z)
        return A, Z

In [77]:
vocabulary["pad"] = 0

### Debug

In [28]:
batch_size = 64
sequence_length = 200
model_dim = 2048
d_model = 512
d_ff = 2048
H = 8
dropout = 0.1

In [29]:
model = AttentionHead(d_model, H)

In [30]:
len(padded_sequences[0])
dim = len(padded_sequences[0])

In [31]:
model

AttentionHead(
  (Wq): Linear(in_features=512, out_features=512, bias=True)
  (Wk): Linear(in_features=512, out_features=512, bias=True)
  (Wv): Linear(in_features=512, out_features=512, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (Wo): Linear(in_features=512, out_features=512, bias=True)
)

In [32]:
x = torch.randn(batch_size, sequence_length, d_model)

In [33]:
Z = model(x)

Z.shape=torch.Size([64, 200, 512])


In [38]:
# len(long_tokenized)

In [39]:
len(vocabulary)

42896

In [40]:
# long_tokenized[:2].shape

In [41]:
# long_tokenized.shape

In [34]:
from sklearn.model_selection import train_test_split

In [43]:
# len(vocabulary.values())

## Feed Forward and Attention Residual

In [44]:
batch, sentence_length, embedding_dim = 20, 10, 8
embedding = torch.randn(batch, sentence_length, embedding_dim)
layer_norm = nn.LayerNorm(embedding_dim)

In [45]:
x = layer_norm(embedding)

In [46]:
x

tensor([[[ 7.9830e-01,  1.8717e+00, -8.7288e-01,  ...,  1.8639e-01,
          -1.7142e+00,  1.4794e-01],
         [-5.3054e-01,  1.0776e+00,  3.1714e-01,  ...,  1.2609e+00,
           4.5796e-01, -2.1307e+00],
         [-6.5479e-01, -1.8833e+00,  1.0518e-01,  ..., -2.7573e-01,
           4.7722e-01,  1.0266e+00],
         ...,
         [-1.5752e+00,  1.5958e+00,  6.7133e-01,  ..., -1.1235e+00,
           9.2012e-01,  1.1861e-01],
         [ 4.5227e-01, -1.7416e+00, -6.2634e-01,  ...,  3.3873e-01,
          -1.1805e+00,  6.2863e-01],
         [ 1.0172e-01,  3.0511e-03, -1.9878e+00,  ...,  3.0186e-01,
           1.9227e-01, -1.0028e+00]],

        [[-1.1427e+00, -7.6485e-01, -4.5299e-01,  ..., -2.4802e-01,
           1.7758e+00,  1.5463e+00],
         [ 3.9634e-01,  4.1909e-01, -9.3178e-02,  ...,  9.6263e-01,
           2.6430e-01,  8.9448e-01],
         [ 9.5739e-01,  1.0450e+00,  1.4010e-01,  ..., -1.5597e+00,
           8.3804e-01,  7.0561e-01],
         ...,
         [ 1.0857e+00, -7

In [47]:
embedding = nn.Embedding(10, 3, padding_idx=0)

In [48]:
embedding_sample = nn.Embedding(10, 3, padding_idx=0)

In [49]:
input = torch.LongTensor([[0, 2, 0, 5]])

In [50]:
input.shape

torch.Size([1, 4])

In [51]:
embedding_sample = embedding_sample(input)

In [52]:
embedding_sample

tensor([[[ 0.0000,  0.0000,  0.0000],
         [-1.3114,  1.1112,  2.0428],
         [ 0.0000,  0.0000,  0.0000],
         [-0.5806, -2.3103, -1.0878]]], grad_fn=<EmbeddingBackward0>)

In [53]:
class FFN(nn.Sequential):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.layer1 = nn.Linear(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_ff)
        self.activation = nn.GELU()
        self.out = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.layer1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.out(x)
        return x

In [54]:
ffn = FFN(d_model, d_ff)

In [55]:
ffn

FFN(
  (layer1): Linear(in_features=512, out_features=2048, bias=True)
  (norm1): LayerNorm((2048,), eps=1e-05, elementwise_affine=True, bias=True)
  (activation): GELU(approximate='none')
  (out): Linear(in_features=2048, out_features=512, bias=True)
)

In [56]:
ffn = nn.Sequential(
    nn.Linear(d_model, d_ff), nn.LayerNorm(d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
)

In [57]:
x.shape, d_ff, d_model

(torch.Size([20, 10, 8]), 2048, 512)

In [58]:
class Transformer(nn.Module):
    def __init__(self, d_model, d_ff, num_head, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.H = num_head
        self.dropout = nn.Dropout(p=dropout)

        self.attention = AttentionHead(d_model, num_head, dropout)
        self.ffn = ffn(d_model, d_ff)

    def forward(self, x, attn_mask=False):
        Z = self.attention(x, attn_mask)
        Z = self.dropout(Z)
        Z = Z + x
        Z = self.ffn(Z) + x
        return Z

In [59]:
corpus = X_train

In [60]:
embedding = nn.Embedding(10, 3, padding_idx=0)
input = torch.LongTensor([[0, 2, 0, 5]])
embedding(input).shape

torch.Size([1, 4, 3])

## Encoder Decoder

In [78]:
d_in = len(corpus)

In [79]:
# encoder = nn.Linear(long_tokenized.shape[0],long_tokenized.shape[1])

In [80]:
embedding

Embedding(3, 5, max_norm=1.0)

### Constants

In [238]:
VOCAB_SIZE = len(vocabulary)
BATCH_SIZE = 64

embedding_dim = 256
# no of GRUs
units = 1024
EPOCHS = 30
vocab_inp_size = len(vocabulary)
vocabulary_target_size = len(vocabulary_target)

In [82]:
len(train_index)

42897

### Data

In [62]:
from torch.utils.data import Dataset, DataLoader

In [63]:
dataset = TensorDataset(X_train, y_train)

In [64]:
X_train[0]

tensor([1, 2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [65]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]),
 tensor([1, 2, 3, 4, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0]))

In [70]:
# todo convert to graph
class Hindi_English(Dataset):
    def __init__(self):
        pass

In [67]:
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=2)

In [68]:
x, y = next(iter(data_loader))

In [69]:
len(data_loader)

1322

### Gru Operation

In [151]:
n, d, m = 3, 5, 7
embedding = nn.Embedding(n, d, max_norm=1.0)
print(f"{embedding.weight.shape=}")
W = torch.randn((m, d), requires_grad=True)
print(f"{W.shape=}")
idx = torch.tensor([1, 2])
a = (
    embedding.weight.clone() @ W.t()
)  # weight must be cloned for this to be differentiable
print(f"{a.shape=},{(a.unsqueeze(0)).shape=}")
b = embedding(idx) @ W.t()  # modifies weight in-place
print(f"{b.shape},{(b.unsqueeze(1)).shape=}")
out = a.unsqueeze(0) + b.unsqueeze(1)
loss = out.sigmoid().prod()
print(loss.backward())

embedding.weight.shape=torch.Size([3, 5])
W.shape=torch.Size([7, 5])
a.shape=torch.Size([3, 7]),(a.unsqueeze(0)).shape=torch.Size([1, 3, 7])
torch.Size([2, 7]),(b.unsqueeze(1)).shape=torch.Size([2, 1, 7])
None


In [72]:
out.shape

torch.Size([2, 3, 7])

In [152]:
rnn = nn.GRU(input_size=10, hidden_size=20, num_layers=2)
input = torch.randn(2, 3, 10)
embedding_inp = nn.Embedding()
h0 = torch.randn(2, 3, 20)  # first if batch size or number of layers
output, hn = rnn(input, h0)
print(f"{output.shape=},{h0.shape=}")

output.shape=torch.Size([2, 3, 20]),h0.shape=torch.Size([2, 3, 20])


### Encoder

In [56]:
import numpy as np


class Encoder(nn.Module):
    def __init__(self, batch_sz, embedding_dim, enc_units, vocab_size):
        super().__init__()
        self.batch_sz = batch_sz  # set batch size
        self.enc_units = enc_units  # set the number of GRU units
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim)
        # self.hidden_state=256
        self.gru = nn.GRU(embedding_dim, self.enc_units, batch_first=True)

    def forward(self, x, hidden=None):
        print(f"{x.shape}")
        x = self.embedding_layer(x)
        print(f"Embedded: {x.shape=}")
        # assert h==sample_hidden
        output, state = self.gru(x, hidden)
        return output, state

    def initialize_hidden(self):
        return torch.zeros(1, self.batch_sz, self.enc_units)

#### Debug

### Decoder

In [176]:
class Decoder(nn.Module):
    def __init__(self, batch_sz, embedding_dim, dec_units, vocab_size):
        super().__init__()
        self.batch_sz = batch_sz  # batch_size which is defined as 64
        self.dec_units = dec_units  # the number of decoder GRU units
        self.embedding_layer = nn.Embedding(
            vocab_size,
            embedding_dim,
        )
        self.gru = nn.GRU(embedding_dim, self.dec_units, batch_first=True)
        self.out = nn.Linear(dec_units, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding_layer(x)
        output, state = self.gru(x, hidden)
        print(f"{output.shape=},{state.shape=}")

        logits = self.out(output)
        return logits, state

### Debug


In [57]:
encoder = Encoder(BATCH_SIZE, embedding_dim, units, vocab_inp_size)

In [58]:
encoder

Encoder(
  (embedding_layer): Embedding(49763, 256)
  (gru): GRU(256, 1024, batch_first=True)
)

In [102]:
sample_hidden = encoder.initialize_hidden()

In [103]:
print(f"f{sample_hidden.shape=}")

fsample_hidden.shape=torch.Size([1, 64, 1024])


In [95]:
65536 / 64 / 16

64.0

In [106]:
out, state = encoder(x)
print(f"{out.shape=},{state.shape=}")

torch.Size([64, 56])
Embedded: x.shape=torch.Size([64, 56, 256])
out.shape=torch.Size([64, 56, 1024]),state.shape=torch.Size([1, 64, 1024])


In [175]:
x.shape

torch.Size([64, 56])

In [107]:
out1, state1 = encoder(x, sample_hidden)
print(f"{out1.shape=},{state1.shape=}")

torch.Size([64, 56])
Embedded: x.shape=torch.Size([64, 56, 256])
out1.shape=torch.Size([64, 56, 1024]),state1.shape=torch.Size([1, 64, 1024])


In [177]:
decoder = Decoder(BATCH_SIZE, 1024, units, len(vocabulary_target))

In [178]:
sample_hidden = encoder.initialize_hidden()

In [179]:
sample_hidden.shape

torch.Size([1, 64, 1024])

In [180]:
x.shape

torch.Size([64, 56])

In [199]:
import numpy as np

uniform = np.random.uniform(1, (BATCH_SIZE, 1))
print(f"{uniform.shape=}")
x1 = torch.rand(BATCH_SIZE, 2).uniform_(-3, 3)
x1

uniform.shape=(2,)


tensor([[ 2.9310, -2.2455],
        [ 2.0715,  1.4720],
        [ 0.1594,  2.8651],
        [-2.3500, -1.9155],
        [ 1.7964, -1.6437],
        [ 0.3484,  1.1460],
        [ 2.6716, -2.3896],
        [ 1.6730,  1.0781],
        [-2.1669,  0.6740],
        [-2.5367,  1.7841],
        [-1.1372,  0.2348],
        [-1.0199, -1.4500],
        [-2.6651,  1.9408],
        [-2.5371,  0.3945],
        [-0.4912,  1.5056],
        [ 2.7938, -2.9537],
        [ 0.5910,  0.5767],
        [-0.2063, -2.0336],
        [ 2.8707,  0.6995],
        [ 0.8520,  2.9245],
        [-0.3325,  0.3486],
        [-0.5837,  0.4983],
        [-0.8480, -2.4602],
        [-0.0765,  1.9490],
        [-2.2148, -0.6662],
        [ 1.9251, -0.5664],
        [ 2.9517, -0.9166],
        [ 0.6620,  1.5075],
        [-1.2217, -1.6449],
        [ 1.4825,  1.8838],
        [-1.4367,  0.5195],
        [-2.7354, -2.0919],
        [-1.7562,  2.2854],
        [-0.6869,  0.7093],
        [-2.1390,  2.7204],
        [ 1.4134, -0

In [182]:
uniform.ndim

1

In [206]:
encoder

Encoder(
  (embedding_layer): Embedding(49763, 256)
  (gru): GRU(256, 1024, batch_first=True)
)

In [207]:
list(encoder.named_modules())

[('',
  Encoder(
    (embedding_layer): Embedding(49763, 256)
    (gru): GRU(256, 1024, batch_first=True)
  )),
 ('embedding_layer', Embedding(49763, 256)),
 ('gru', GRU(256, 1024, batch_first=True))]

In [183]:
x1.ndim

2

In [184]:
x1 = torch.tensor(x1.detach().clone(), dtype=torch.long)

/var/folders/px/m0g9wbyn1sv678fsx9lgsm600000gn/T/ipykernel_29433/2154946776.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x1=torch.tensor(x1.detach().clone(),dtype=torch.long)


In [185]:
x1.ndim

2

In [200]:
sample_decoder_output, state2 = decoder(torch.tensor(uniform, dtype=torch.long))
print(sample_decoder_output.shape, state2.shape)

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])
torch.Size([2, 40397]) torch.Size([1, 1024])


In [202]:
decoder

Decoder(
  (embedding_layer): Embedding(40397, 1024)
  (gru): GRU(1024, 1024, batch_first=True)
  (out): Linear(in_features=1024, out_features=40397, bias=True)
)

In [208]:
sample_decoder_output2, state2 = decoder(
    torch.tensor(np.random.uniform(3, (BATCH_SIZE, 3)), dtype=torch.long), sample_hidden
)
sample_decoder_output2

RuntimeError: For unbatched 2-D input, hx should also be 2-D but got 3-D tensor

In [210]:
print(sample_decoder_output2.shape, state2.shape)

torch.Size([2, 40397]) torch.Size([1, 1024])


In [211]:
sample_decoder_output1, state1 = decoder(torch.tensor(uniform, dtype=torch.long))

output.shape=torch.Size([2, 1024]),state.shape=torch.Size([1, 1024])


In [ ]:
len(data_loader)

In [212]:
x

tensor([[  1,   2,   3,  ...,   0,   0,   0],
        [  6,   7,   8,  ...,   0,   0,   0],
        [ 12,  13,  14,  ...,   0,   0,   0],
        ...,
        [256,  61, 257,  ...,   0,   0,   0],
        [259,  13, 260,  ...,   0,   0,   0],
        [261, 262,   0,  ...,   0,   0,   0]])

In [ ]:
y

In [ ]:
sample_hidden = encoder.initialize_hidden()

In [ ]:
sample_hidden.shape

In [ ]:
sample_hidden = sample_hidden.view(1, sample_hidden.shape[0], sample_hidden.shape[1])

In [ ]:
x.shape

In [ ]:
x_seq = torch.tensor([[1.0] * 5, [2.0] * 5, [3.0] * 5])
x_seq.shape

In [ ]:
x_seq.reshape(1, 3, 5)

In [ ]:
x.shape

In [ ]:
batch_size

In [ ]:
x_batched = x.view(batch_size, 1, x.shape[-1])

In [ ]:
x_batched.shape

In [ ]:
X_train.shape

In [ ]:
# sample_output, sample_hidden = encoder(x, sample_hidden)

In [ ]:
batch_size

In [ ]:
encoder

## Training

In [222]:
import dataclasses
from dataclasses import dataclass

from dataclasses import dataclass


@dataclass(init=True, repr=True)
class InventoryItem:
    """Class for keeping track of an item in inventory."""

    name: str
    unit_price: float
    quantity_on_hand: int = 0

    def total_cost(self) -> float:
        return self.unit_price * self.quantity_on_hand

In [223]:
item = InventoryItem(name="T shirt", unit_price=10, quantity_on_hand=3)

In [224]:
item

InventoryItem(name='T shirt', unit_price=10, quantity_on_hand=3)

In [225]:
item.total_cost()

30

In [226]:
from typing import List


@dataclass(init=True)
class Result:
    train_loss: List[float]
    test_loss: List[float]

    train_acc: List[float]
    test_acc: List[float]

In [227]:
input = torch.randn(3, 2, requires_grad=True)
target = torch.rand(3, 2, requires_grad=False)
loss = F.binary_cross_entropy(torch.sigmoid(input), target)
loss.backward()

In [228]:
loss.item()

0.7890180945396423

In [229]:
loss.float()

tensor(0.7890, grad_fn=<BinaryCrossEntropyBackward0>)

In [230]:
dir(encoder)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pre_hooks',
 '_maybe_warn_non_full_backward_hook',
 '_modules',
 '_named_members',
 '_non_persistent_buffers_se

In [231]:
encoder = encoder.to(device)

In [233]:
decoder = decoder.to(device)

In [234]:
loss = torch.nn.BCEWithLogitsLoss()
criterion = torch.optim.AdamW(encoder.parameters(), lr=0.3, weight_decay=0.1)

In [239]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

In [237]:
for p, m in encoder.named_parameters():
    print(f"{p},{m.shape=}")

embedding_layer.weight,m.shape=torch.Size([49763, 256])
gru.weight_ih_l0,m.shape=torch.Size([3072, 256])
gru.weight_hh_l0,m.shape=torch.Size([3072, 1024])
gru.bias_ih_l0,m.shape=torch.Size([3072])
gru.bias_hh_l0,m.shape=torch.Size([3072])


In [ ]:
loss_object = torch.nn.BCEWithLogitsLoss()

In [240]:
from tqdm.auto import tqdm
def train_epoch(epoch: int,model: nn.Module,train_loader: torch.utils.data.DataLoader,optim: torch.optim.Optimizer):
    model.train()

    criterion = nn.CrossEntropyLoss()
    for batched_input, batched_target in tqdm(
        train_loader, desc=f"Training @ epoch {epoch}"
    ):
      loss =



